età:imputazione non a valor medio per tutto

Si consideri il data set Titanic composto dai file train.csv e test.csv, allegati al presente compito, costituiti da 891+418 record che descrivono i passeggeri del Titanic e li etichettano come sopravvissuti o meno sulla base delle seguenti caratteristiche:
- PassengerId
- Survived (1/0, solo train)
- Pclass (1, 2, 3)
- Name
- Sex (‘male’, ‘female’)
- Age (frazionaria se meno di 1; se stimata è nella forma xx.5)
- SibSp (numero di familiari; Sibling: fratelli/sorelle; Spouse: moglie/marito)
- Parch (numero familiari; Parent: padre/madre; Child: figli/figlie; Parch=0 per piccoli accompagnati solo dalla tata)
- Ticket
- Fare
- Cabin
- Embarked (C = Cherbourg, Q = Queenstown, S = Southampton)

1. Effettuare uno split 90%-10% tra training set e validation set. Individuare
eventuali dati mancanti e farne l’imputazione secondo criteri che rispecchino
la diversa stratificazione sociale e distribuzione di genere dei passeggeri
ovvero rimuoverle se troppo sparse. Effettuare la codifica delle feature
categoriche e l’eventuale scaling di tutte le feature, una volta trasformate in
numeriche. Verificare la presenza di feature multicollneari e ridurle tramite
opportune combinazioni lineari. Infine, procedere alla feature selection
attraverso una tecnica embedded che impieghi un classificatore come
modello. La scelta del modello embedded è lasciata al candidato, tenendo
conto che il problema è di classificazione binaria.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
import matplotlib.pyplot as plot
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel

train_df = pd.read_csv("C:/Users/giova/Documents/GitHub/BigData2/Condiviso/Giovanni/Simulazione2/train.csv", sep=',')
test_df = pd.read_csv("C:/Users/giova/Documents/GitHub/BigData2/Condiviso/Giovanni/Simulazione2/test_augmented.csv", sep=',')
print(train_df)

X = train_df.drop(columns = ['Survived'])
y = train_df['Survived']

#Split 90-10 per creare il validation set
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)

print(f"Dimensioni del dataset originale: {X.shape[0]}")
print(f"Dimensioni del Training Set (90%): {X_tr.shape[0]}")
print(f"Dimensioni del Validation Set (10%): {X_val.shape[0]}")

#Imputazione dati mancanti
report_missing = (X_tr.isnull().sum() / len(X_tr)) *100
print(f"\nPercentuale dati train mancanti:\n{report_missing}")

train_clean = X_tr.copy()
val_clean = X_val.copy()  #copia dei dataframe per non perdere gli originali

#Calcolo statistiche per prevenire il Data Leakage
age_medians = train_clean.groupby(['Pclass', 'Sex'])['Age'].median()
fare_median = train_clean['Fare'].median()
embarked_mode = train_clean['Embarked'].mode()[0]

datasets = [train_clean, val_clean, test_df]  #applichiamo le stesse trasfornazioni su tutti e tre i dataset

for df in datasets:
    #Drop delle feature troppo sparse o non numericamente utili al modello
    df.drop(columns=['Cabin', 'PassengerId', 'Name', 'Ticket'], inplace=True, errors='ignore')

    #Imputazione: stratificazione per Classe e Genere
    #Age - mediana
    def fill_age(row):
        if pd.isnull(row['Age']):
            return age_medians[row['Pclass']][row['Sex']]
        return row['Age']
    df['Age'] = df.apply(fill_age, axis=1)

    #Fare - mediana
    df['Fare'] = df['Fare'].fillna(fare_median)

    #Embarked - moda
    df['Embarked'] = df['Embarked'].fillna(embarked_mode)


    #Family_size (unione si SibSp + Parch)
    df['Family_size'] = df['SibSp'] + df['Parch']
    df.drop(columns=['SibSp', 'Parch'], inplace=True)

#A questo punto i tre dataframe di partenda saranno puliti


#Encoding e Scaling
num_features = ['Age', 'Family_size', 'Fare']
cat_features = ['Pclass', 'Sex', 'Embarked']

if len(cat_features)>0:
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-5)
    train_clean[cat_features] = encoder.fit_transform(train_clean[cat_features])
    val_clean[cat_features] = encoder.transform(val_clean[cat_features])
    test_df[cat_features] = encoder.transform(test_df[cat_features])
#adesso le feature categoriche sono state convertite in numeriche

#recuperiamo adesso tutte le feature numeriche dal training set pulito
num_features = train_clean.select_dtypes(include='number').columns.tolist()
print(num_features)  #le stamperà tutte e sei

if len(num_features)>0:
    scaler = StandardScaler()
    train_clean[num_features] = scaler.fit_transform(train_clean[num_features])
    val_clean[num_features] = scaler.transform(val_clean[num_features])
    test_df[num_features] = scaler.transform(test_df[num_features])

X_tr_final = train_clean[num_features]
X_val_final = val_clean[num_features]
test_final = test_df[num_features]


#Analisi multicollinearità
corr_matrix = X_tr_final.corr().abs()
corr_threshold = 0.6
corr_pairs = corr_matrix.unstack()
corr_pairs = corr_pairs[corr_pairs != 1.0]
print(corr_pairs)
related_features = corr_pairs[corr_pairs>corr_threshold]
print(f"\n\n{related_features.sort_values(ascending=False)}")  #non risultano feature altamente correlate


#Feature Selection
clf = RandomForestClassifier(random_state=42)
clf.fit(X_tr_final, y_tr)
selector = SelectFromModel(clf, prefit=True)
selected_features = X_tr_final.columns[selector.get_support()]  #feature sopravvissute alla selezione

print(f"\nNumero di feature mantenute: {len(selected_features)} su {len(num_features)}")
print("Feature Selezionate:", selected_features.tolist())

#Matrici definitive
X_tr_selected = selector.transform(X_tr_final)
X_val_selected = selector.transform(X_val_final)
test_selected = selector.transform(test_final)


     PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                             Allen, Mr. William Henry    male  35.0      0   
..                                                 ...     ...   ... 

C:\Users\giova\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
C:\Users\giova\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
C:\Users\giova\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(


2. Implementare un classificatore SVM non lineare per il data set curato risultato
del punto 1 con i seguenti iper-parametri:
- C = {1, 1/sqrt(n_samples)}
- kernel={RBF, polinomiale}
- grado del kernel polinomiale = {3, 4}
Usare l’accuracy come metrica e stampare l’accuracy del miglior classificatore
e la test accuracy ottenuta in predizione.

In [3]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

C_value = 1/np.sqrt(len(X_tr_selected))

param_grid = [
    {
        'C': [1, C_value],
        'kernel': ['rbf'],
    },
    {
        'C': [1, C_value],
        'kernel': ['poly'],
        'degree': [3, 4]
    }
]

clf = SVC(random_state=42)
grid_search_clf = GridSearchCV(
    estimator=clf,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=3
)

grid_search_clf.fit(X_tr_selected, y_tr)
print(f"Miglior accuratezza: {grid_search_clf.best_score_} - Miglior classificatore: {grid_search_clf.best_estimator_}")

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Miglior accuratezza: 0.7952484472049689 - Miglior classificatore: SVC(C=1, random_state=42)


3. Implementare una piccola rete neurale densa con almeno tre layer nascosti in PyTorch, utilizzando la semplice API torchnn.py, per eseguire la classificazione binaria. Si dovrà implementare anche il relativo Dataset per caricare i dati. I layer nascosti dovranno avere un dropout pari a 0.2. Si utilizzi per l’addestramento l’ottimizzatore Adam con weight decay pari a 1x10-4 e learning rate dercescente esponenzialmente a partire da 0.01. Implementare una callback di early stopping con una pazienza sulla validation loss di 5 epoche e un incremento minimo di miglioramento pari a 0.01; implementare anche una callback di model checkpoint per il salvataggio del solo miglior modello rispetto alla minima validation loss.

In [14]:
import torch
import torch.optim as optim
from torch import nn
from torchvision import datasets
from torchvision.transforms import v2
from torch.utils.data import TensorDataset
from torchnn import *

class NeuralNetwork(nn.Module):
    def __init__(self, features_input):
        super().__init__()  #richiama il costruttore di nn.Module

        self.network = nn.Sequential(
            #primo layer nascosto, a cui passo il numero di feature in ingresso e i neuroni in esso
            nn.Linear(features_input, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            #secondo layer
            nn.Linear(256, 128),  #numero di neuroni nel primo layer e lo stesso numero dimezzato
            nn.ReLU(),
            nn.Dropout(0.2),
            #terzo layer
            nn.Linear(128, 64),  #numero di neuroni nel secondo layer e lo stesso numero dimezzato
            nn.ReLU(),
            nn.Dropout(0.2),
            #layer di output
            nn.Linear(64, 2), #numero di neurono nel terzo layer e numero di classi per la classificazione (in questo caso binaria -> 2)
        )

    def forward(self, x):
        return self.rete(x)
    

model = NeuralNetwork(len(X_tr_selected)).to(device='cuda' if torch.cuda.is_available() else 'cpu')
print(model)

#creo i tensori a partire dai miei dataset
X_tr_tensor = torch.tensor(X_tr_selected, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_selected, dtype=torch.float32)
y_tr_tensor = torch.tensor(y_tr.values, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32)

train_dataset = TensorDataset(X_tr_tensor, y_tr_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)


#adesso costruisco i dataloader
train_dataloader, val_dataloader, test_dataloader = make_dataloaders(train_dataset, val_dataset, val_dataset, batch=64)

loss_fn = nn.CrossEntropyLoss()  #funzione di loss da applicare al modello neurale

opt = optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)  #ottimizzatore Adam

#Model Checkpoint per il salvataggio del modello migliore rispetto alla minima validation loss
class ModelCheckpoint(EarlyStopping):
    def __init__(self, patience, min_delta, save_path):
        super().__init__(patience=patience, min_delta=min_delta)
        self.save_path = save_path

    def salva_modello(self, validation_loss, model, opt, epoch):
        super.__call__(validation_loss)  #richiama il check dell'early stopping
        if validation_loss == self.min_validation_loss:
            save_model(model, 
                       optimizer=opt, 
                       current_epoch=epoch, 
                       train_loss=[], 
                       val_loss=[], 
                       test_loss=[], 
                       accuracy=[], 
                       metrics={}, 
                       path=self.save_path)  #teniamo traccia del percorso di salvataggio
            print(f"Validation loss migliorata: {validation_loss}, modello salvato\n")


early_stopping_callback = EarlyStopping(patience=5, min_delta=0.01)

print("Inizio addestramento Rete Neurale...")
train_loss, val_loss, test_loss, accuracy, metrics = train_test(
    model=model,
    optimizer=opt,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    test_dataloader=test_dataloader,
    epochs=30,
    train_loss_fn=loss_fn,
    test_loss_fn=loss_fn,
    early_stopping=early_stopping_callback
)



NeuralNetwork(
  (network): Sequential(
    (0): Linear(in_features=801, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=128, out_features=64, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.2, inplace=False)
    (9): Linear(in_features=64, out_features=2, bias=True)
  )
)
Shape e tipo dei campioni: torch.Size([64, 3]), torch.float32
Shape e tipo delle etichette: torch.Size([64]) torch.float32
Inizio addestramento Rete Neurale...


Epoch    1:   0%|          | 0/13 [00:00<?, ?it/s]

AttributeError: 'NeuralNetwork' object has no attribute 'rete'

4. Conservare la lista delle accuracy di addestramento e di test su tutte le epoche del classificatore neurale e stamparne il grafico. Confrontare i risultati del miglior classificatore SVM e del classificatore neurale calcolando e
stampando, per ciascuno, la matrice di confusione, il valore di accuracy e di loss. Calcolare e stampare in un unico grafico le curve ROC dei due classificatori binari e stamparne anche le relative AUC.